# untwisting_cables_10105: next 1 hour

This notebook matches `05_model_training/untwisting_cables_10105/01_horizon_10min/03_train_selected_features_variant.ipynb` but forecasts the **next 1 hour**. Its alarm target and feature settings were copied from that notebook. It uses **an editable selected sensor subset**, taking only the **previous hour** of readings, and issues forecasts every **10 minutes**.

Edit `ALARM_CODES`, `TARGET_NAME`, `TURBINE_IDS` and, if present, `SELECTED_FEATURES` in settings. Run independently after preprocessing; earlier horizon notebooks are not prerequisites. The five models, chronological splits, validation-based threshold selection, evaluation tables, confusion matrices and prediction-history plots are included.

Reports are separated by alarm target, horizon and feature selection where applicable. Longer horizons create overlapping positive windows around a single event. Split boundaries and label coverage use this notebook's horizon. Training-period predictions are in-sample. The selected-feature variant is retained to match the 10-minute folder; identical settings use the same selected-feature report directory.


## Alarm-code reference

Descriptions transcribed from the supplied alarm-code table. Its stopping indicator is **1** for every code listed below. This reference does not change the selected training target; edit `ALARM_CODES` in settings to choose codes.

| Alarm code | Description |
|---|---|
| 1005 | Availability - low wind |
| 3130 | Pitch lubrication |
| 8000 | Windspeed too high to operate |
| 8210 | Stopped, due to icing |
| 8230 | Ice detection: Low torque |
| 8234 | Ice detection: No cut in |
| 8235 | Ice Detect: Stopped |
| 8236 | Ice Detect: Stopped, De-Ice |
| 10105 | Stopped, untwisting cables |


In [ ]:
# Run this first to see sensor names you can copy into SELECTED_FEATURES below.
from pathlib import Path
import pyarrow.parquet as pq

_feature_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                      if (p / 'notebooks' / '01_data_export/01_export_yearly_turbine_scada.ipynb').is_file()), None)
if _feature_root is None:
    raise FileNotFoundError('Open this notebook from Iteration2 or its notebooks folder.')
_feature_files = sorted((_feature_root / 'Data_Parquets_10min').glob('*/Station_*/data.parquet'))
if not _feature_files:
    raise FileNotFoundError('No processed Parquet files found. Run notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb first.')
_available_features = sorted({name for path in _feature_files
                              for name in pq.read_schema(path).names if name.startswith('wtc_')})
print(f'{len(_available_features)} available sensor features (across {len(_feature_files)} turbine-years):')
print('\n'.join(_available_features))
print('\nCopy the names you want into SELECTED_FEATURES in the settings cell below.')
print('This lists column names; some sensors may have missing readings for particular years/turbines.')


In [ ]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (average_precision_score, precision_recall_curve, precision_score,
    recall_score, f1_score, fbeta_score, roc_auc_score, balanced_accuracy_score,
    matthews_corrcoef, brier_score_loss, confusion_matrix, ConfusionMatrixDisplay)
from lightgbm import LGBMClassifier
import joblib

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'notebooks' / '01_data_export/01_export_yearly_turbine_scada.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run from Iteration2 or its notebooks folder.')
# Any listed code starting in the horizon makes the grouped target positive.
ALARM_CODES = ['10105']
TARGET_NAME = 'untwisting_cables'  # Stopped, untwisting cables
if not isinstance(ALARM_CODES, (list, tuple, set)) or not ALARM_CODES:
    raise ValueError('ALARM_CODES must be a nonempty collection of alarm-code strings.')
ALARM_CODES = sorted({str(code).strip() for code in ALARM_CODES})
if any(not code.isdigit() for code in ALARM_CODES):
    raise ValueError('Alarm codes must contain digits only.')
if not TARGET_NAME or any(not (c.isalnum() or c in '_-') for c in TARGET_NAME):
    raise ValueError('TARGET_NAME must contain only letters, digits, underscores or hyphens.')
ALARM_CODE = ' / '.join(ALARM_CODES)  # Display text, derived from the group above.
TARGET_ID = TARGET_NAME + '_' + '-'.join(ALARM_CODES)
TURBINE_IDS = ['2304511']
HORIZON = pd.Timedelta(hours=1)  # Forecast the next 1 hour.
LOOKBACK = pd.Timedelta(hours=1)
STEP = pd.Timedelta(minutes=10)
TRAIN_END = pd.Timestamp('2023-01-01')
VALIDATION_END = pd.Timestamp('2024-01-01')
SEED = 42
# EDIT THIS LIST to choose exactly which raw sensors the models may use.
SELECTED_FEATURES = [
    'wtc_GearPres_mean'
]

# SELECTED_FEATURES = [
#     'wtc_HydPress_mean',
#     'wtc_HubPresA_mean',
#     'wtc_HubPresB_mean',
#     'wtc_HubPresC_mean',
# ]
ROLLING_STATISTICS = ['mean', 'std', 'min', 'max']
if not isinstance(SELECTED_FEATURES, (list, tuple)) or not SELECTED_FEATURES:
    raise ValueError('SELECTED_FEATURES must be a nonempty list of sensor names.')
if any(not isinstance(c, str) or not c.startswith('wtc_') for c in SELECTED_FEATURES):
    raise ValueError('Choose sensor columns starting with wtc_; alarm flags and labels are not inputs.')
if len(set(SELECTED_FEATURES)) != len(SELECTED_FEATURES):
    raise ValueError('Remove duplicate SELECTED_FEATURES entries.')
if (not ROLLING_STATISTICS or len(set(ROLLING_STATISTICS)) != len(ROLLING_STATISTICS)
        or any(c not in {'mean', 'std', 'min', 'max'} for c in ROLLING_STATISTICS)):
    raise ValueError('Choose distinct statistics from mean, std, min and max.')
import hashlib
selection_config = {'sensors': SELECTED_FEATURES, 'statistics': ROLLING_STATISTICS,
                    'turbines': TURBINE_IDS, 'train_end': str(TRAIN_END),
                    'validation_end': str(VALIDATION_END), 'seed': SEED}
SELECTION_ID = hashlib.sha256(json.dumps(selection_config, sort_keys=True).encode()).hexdigest()[:10]
OUTPUT = ROOT / 'reports' / f'alarm_{TARGET_ID}_{int(HORIZON.total_seconds() // 60)}min_selected_{SELECTION_ID}'
print('Selected raw sensors:', SELECTED_FEATURES)
print('Rolling statistics:', ROLLING_STATISTICS)
print('Reports:', OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
assert HORIZON > pd.Timedelta(0) and LOOKBACK == pd.Timedelta(hours=1)

import sys
if str(ROOT / "notebooks" / "shared") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks" / "shared"))
from interval_parquet import load_aligned
PROCESSED_ROOT = ROOT / "Data_Parquets_10min"

# Plot labels always follow the chosen ALARM_CODES (including single-code targets).
ALARM_DESCRIPTIONS = {
    '1005': 'Availability - low wind', '3130': 'Pitch lubrication',
    '8000': 'Windspeed too high to operate', '8210': 'Stopped, due to icing',
    '8230': 'Ice detection: Low torque', '8234': 'Ice detection: No cut in',
    '8235': 'Ice Detect: Stopped', '8236': 'Ice Detect: Stopped, De-Ice',
    '10105': 'Stopped, untwisting cables',
}
import textwrap

def alarm_plot_label():
    description = '; '.join(f'{code}: {ALARM_DESCRIPTIONS.get(code, "Description unavailable")}'
                            for code in ALARM_CODES)
    meaning = 'ANY of these codes starts' if len(ALARM_CODES) > 1 else 'This alarm starts'
    return '\n'.join(textwrap.wrap(meaning + ' | ' + description, width=115))



In [ ]:
# Only prepared, regular interval data is read; no Excel/raw conversion here.
if not PROCESSED_ROOT.is_dir():
    raise FileNotFoundError('Run 03_preprocessing/01_preprocess_10min_parquets.ipynb first.')
paths = {}
for directory in PROCESSED_ROOT.iterdir():
    if not directory.name.isdigit():
        continue
    for turbine in TURBINE_IDS:
        folder = directory / f'Station_{turbine}'
        if (folder / 'data.parquet').is_file():
            paths[(turbine, int(directory.name))] = folder
if not paths:
    raise FileNotFoundError('No selected turbine data. Run notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb first.')
print('Available turbine-years:', sorted(paths))


In [ ]:
# Available sensor names for editing SELECTED_FEATURES above.
# Schema inspection only; feature choice and preprocessing never use test labels.
import pyarrow.parquet as pq
available_names = set()
for folder in paths.values():
    available_names.update(name for name in pq.read_schema(folder / 'data.parquet').names
                           if name.startswith('wtc_'))
feature_catalogue = pd.DataFrame({'Sensor column': sorted(available_names)})
feature_catalogue['Selected'] = feature_catalogue['Sensor column'].isin(SELECTED_FEATURES)
display(feature_catalogue)
unknown = set(SELECTED_FEATURES) - available_names
if unknown:
    raise ValueError(f'Unknown selected sensors: {sorted(unknown)}')


In [ ]:
def build_turbine(turbine, folders):
    sensor_parts, event_parts = [], []
    for year, folder in sorted(folders):
        frame, a = load_aligned(PROCESSED_ROOT, year, turbine)
        # Inserted rows are not real sensor observations; metadata is never an input.
        missing = [name for name in SELECTED_FEATURES if name not in frame.columns]
        if missing:
            raise ValueError(f'{turbine}/{year}: selected sensors unavailable: {missing}. Check the catalogue.')
        s = frame.loc[frame.sensorObserved, SELECTED_FEATURES].copy()
        if not isinstance(s.index, pd.DatetimeIndex) or s.index.tz is not None:
            raise ValueError('Expected a naive timestamp index in sensor Parquet.')
        s = s.loc[(s.index >= pd.Timestamp(year, 1, 1)) & (s.index < pd.Timestamp(year + 1, 1, 1))]
        sensor_parts.append(s)
        a = a.loc[a.Alarmcode.astype(str).str.strip().isin(ALARM_CODES)].copy()
        event_parts.append(a.loc[a.TimeOn.ge(pd.Timestamp(year, 1, 1)) & a.TimeOn.lt(pd.Timestamp(year + 1, 1, 1))])
    sensors = pd.concat(sensor_parts).sort_index()
    sensors = sensors.loc[~sensors.index.duplicated(keep='last')]
    cols = list(SELECTED_FEATURES)
    sensors = sensors[cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if ((sensors.index.minute % 10 != 0) | (sensors.index.second != 0) | (sensors.index.microsecond != 0)).any():
        raise ValueError('Off-grid sensors need an explicit alignment decision.')
    grid = pd.date_range(sensors.index.min(), sensors.index.max(), freq=STEP)
    present = pd.Series(grid.isin(sensors.index), index=grid)
    sensors = sensors.reindex(grid)
    # shift(1) guarantees the current reading is not in the preceding-hour window.
    prior = sensors.shift(1).rolling(6, min_periods=1)
    feature_parts = []
    for operation in ROLLING_STATISTICS:
        part = getattr(prior, operation)()
        part.columns = [f'{c}__{operation}' for c in cols]
        feature_parts.append(part)
    features = pd.concat(feature_parts, axis=1).astype('float32')
    coverage = present.shift(1, fill_value=False).rolling(6, min_periods=6).sum().eq(6)
    # Require labels from loaded years for the full horizon, without inspecting
    # future sensor presence or values when deciding whether to issue a prediction.
    steps = int(np.ceil(HORIZON / STEP))
    label_coverage = pd.Series(grid.year.isin({year for year, _ in folders}), index=grid)
    future_coverage = label_coverage.iloc[::-1].rolling(steps, min_periods=steps).sum().iloc[::-1].eq(steps)
    events = pd.concat(event_parts, ignore_index=True)
    events['TimeOn'] = pd.to_datetime(events.TimeOn)
    events['TimeOff'] = pd.to_datetime(events.TimeOff)
    starts = np.sort(events.TimeOn.dropna().to_numpy(dtype='datetime64[ns]'))
    times = grid.to_numpy(dtype='datetime64[ns]')
    y = (np.searchsorted(starts, times + HORIZON.to_timedelta64(), side='left')
         - np.searchsorted(starts, times, side='left') > 0).astype('int8')
    # Exclude predictions while ANY group member is active (new grouped episode prediction).
    active = np.zeros(len(grid) + 1, dtype='int32')
    for event in events.itertuples():
        if pd.isna(event.TimeOn):
            continue
        off = event.TimeOff
        if pd.isna(off) or off <= event.TimeOn:
            off = event.TimeOn.floor('10min') + STEP
        left = grid.searchsorted(event.TimeOn, side='right')
        right = grid.searchsorted(off, side='left')
        if right > left:
            active[left] += 1; active[right] -= 1
    keep = coverage.to_numpy() & future_coverage.to_numpy() & (np.cumsum(active[:-1]) == 0)
    features = features.loc[keep].copy()
    features['target'] = y[keep]
    features['turbine'] = turbine
    features.index.name = 'TimeStamp'
    print(turbine, 'rows:', len(features), 'positive windows:', int(features.target.sum()), flush=True)
    return features

parts = []
for turbine in TURBINE_IDS:
    folders = [(year, folder) for (station, year), folder in paths.items() if station == turbine]
    if not folders:
        raise ValueError(f'No prepared data for turbine {turbine}')
    parts.append(build_turbine(turbine, folders))
data = pd.concat(parts).sort_index()
del parts
feature_columns = [c for c in data if c not in ['target', 'turbine']]
# Purge labels reaching the next split and histories crossing into prior splits.
train = data.loc[data.index + HORIZON <= TRAIN_END]
validation = data.loc[(data.index - LOOKBACK >= TRAIN_END) & (data.index + HORIZON <= VALIDATION_END)]
test = data.loc[data.index - LOOKBACK >= VALIDATION_END]
for name, frame in [('train', train), ('validation', validation), ('test', test)]:
    if frame.empty or frame.target.nunique() < 2:
        raise ValueError(f'{name} needs both classes. Adjust years/turbines, not random splitting.')
display(pd.DataFrame([{'Split': name, 'Rows': len(frame), 'Positives': int(frame.target.sum()),
    'Prevalence': frame.target.mean(), 'First': frame.index.min(), 'Last': frame.index.max()}
    for name, frame in [('train', train), ('validation', validation), ('test', test)]]))
X_train, y_train = train[feature_columns], train.target
X_val, y_val = validation[feature_columns], validation.target
X_test, y_test = test[feature_columns], test.target

expected_features = {f'{sensor}__{stat}' for sensor in SELECTED_FEATURES for stat in ROLLING_STATISTICS}
assert set(feature_columns) == expected_features, 'Unexpected model input column.'
print(f'{len(SELECTED_FEATURES)} sensors -> {len(feature_columns)} rolling model inputs')
print('Model inputs:', feature_columns)
empty_inputs = X_train.columns[X_train.isna().all()].tolist()
if empty_inputs:
    warnings.warn(f'No training observations for these inputs; imputer will fill them with zero: {empty_inputs}')


In [ ]:
# Independent models; preprocessing is fit on training data only.
models = {
 'Constant prevalence': DummyClassifier(strategy='prior'),
 'Logistic regression': make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True),
    StandardScaler(), FunctionTransformer(np.asarray, kw_args={'dtype': np.float64}), LogisticRegression(class_weight='balanced', solver='newton-cholesky', max_iter=100, random_state=SEED)),
 'LightGBM': make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True),
    LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, max_depth=6,
                   class_weight='balanced', random_state=SEED, n_jobs=4, verbosity=-1)),
 'Random forest': make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True),
    RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=10,
                           class_weight='balanced_subsample', random_state=SEED, n_jobs=4))}
validation_scores, test_scores, fit_seconds = {}, {}, {}
for name, model in models.items():
    print('Training', name, flush=True)
    began = time.perf_counter()
    model.fit(X_train, y_train)
    validation_scores[name] = model.predict_proba(X_val)[:, 1]
    test_scores[name] = model.predict_proba(X_test)[:, 1]
    fit_seconds[name] = time.perf_counter() - began

# Fixed random projection: random weights never fitted to target labels.
random_preprocessor = make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), StandardScaler())
random_preprocessor.fit(X_train)
rng = np.random.default_rng(SEED)
random_weights = rng.normal(size=len(feature_columns)) / np.sqrt(len(feature_columns))
for frame, destination in [(X_val, validation_scores), (X_test, test_scores)]:
    z = np.clip(random_preprocessor.transform(frame) @ random_weights, -30, 30)
    destination['Random-weight score'] = 1 / (1 + np.exp(-z))
fit_seconds['Random-weight score'] = np.nan
print('All five models scored.')


In [ ]:
# Select thresholds using validation only, then evaluate untouched test scores.
thresholds = {}
for name, scores in validation_scores.items():
    if name == 'Constant prevalence':
        thresholds[name] = 0.5
        continue
    precision, recall, candidates = precision_recall_curve(y_val, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    thresholds[name] = float(candidates[np.argmax(f1)])

rows = []
for split, truth, scores_by_model, frame in [('Validation', y_val, validation_scores, validation), ('Test', y_test, test_scores, test)]:
    for name, scores in scores_by_model.items():
        pred = scores >= thresholds[name]
        tn, fp, fn, tp = confusion_matrix(truth, pred, labels=[0, 1]).ravel()
        rows.append({'Split': split, 'Model': name, 'Threshold': thresholds[name],
            'Precision': precision_score(truth, pred, zero_division=0),
            'Recall': recall_score(truth, pred, zero_division=0),
            'F1': f1_score(truth, pred, zero_division=0),
            'F2': fbeta_score(truth, pred, beta=2, zero_division=0),
            'PR-AUC (AP)': average_precision_score(truth, scores),
            'Prevalence': truth.mean(), 'ROC-AUC': roc_auc_score(truth, scores),
            'Balanced accuracy': balanced_accuracy_score(truth, pred),
            'MCC': matthews_corrcoef(truth, pred), 'Brier score': brier_score_loss(truth, scores),
            'False positives / observed turbine-day': fp / (len(frame) * STEP / pd.Timedelta(days=1)),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})
metrics = pd.DataFrame(rows)
display(metrics.round(5))
print('PR-AUC is average precision (AP). Compare it with prevalence; accuracy alone is misleading for rare alarms.')
print('Class-weighted and random-weight scores are not calibrated probabilities; interpret Brier scores cautiously.')
print('False-positive rate uses eligible observed windows, not full calendar time.')
metrics.to_csv(OUTPUT / 'metrics.csv', index=False)


In [ ]:
# Test precision-recall curves and one confusion matrix per model.
fig, ax = plt.subplots(figsize=(9, 5))
for name, scores in test_scores.items():
    p, r, _ = precision_recall_curve(y_test, scores)
    ax.plot(r, p, label=f'{name} (AP={average_precision_score(y_test, scores):.4f})')
ax.axhline(y_test.mean(), color='grey', linestyle='--', label='Test prevalence')
ax.set(xlabel='Recall', ylabel='Precision', title=f'Alarm code(s) {ALARM_CODE}: test precision-recall')
ax.legend(fontsize=8); ax.grid(alpha=.2)
fig.suptitle(alarm_plot_label(), fontsize=10)
fig.tight_layout(rect=(0, 0, 1, .87))
fig.savefig(OUTPUT / 'precision_recall.png', dpi=150); plt.show()
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, scores) in zip(axes.flat, test_scores.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, scores >= thresholds[name], labels=[0, 1],
        display_labels=['No selected start', 'Selected start'], ax=ax, colorbar=False, values_format='d')
    ax.set_title(f'{name}\nCode(s): {ALARM_CODE}', fontsize=9)
axes.flat[-1].axis('off')
fig.suptitle(alarm_plot_label(), fontsize=10)
fig.tight_layout(rect=(0, 0, 1, .9))
fig.savefig(OUTPUT / 'confusion_matrices.png', dpi=150); plt.show()


In [ ]:
# Save trained artefacts and reproducibility settings.
joblib.dump({'models': models, 'random_preprocessor': random_preprocessor,
             'random_weights': random_weights, 'thresholds': thresholds,
             'alarm_codes': ALARM_CODES, 'target_name': TARGET_NAME, 'feature_columns': feature_columns, 'selected_features': SELECTED_FEATURES,
             'rolling_statistics': ROLLING_STATISTICS}, OUTPUT / 'trained_models.joblib')
config = {'alarm_codes': ALARM_CODES, 'target_name': TARGET_NAME, 'turbines': TURBINE_IDS, 'horizon': str(HORIZON),
          'lookback': str(LOOKBACK), 'train_end': str(TRAIN_END),
          'validation_end': str(VALIDATION_END), 'seed': SEED,
          'input_folders': [str(p) for p in paths.values()],
          'selected_features': SELECTED_FEATURES, 'rolling_statistics': ROLLING_STATISTICS,
          'selection_id': SELECTION_ID}
(OUTPUT / 'configuration.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
predictions = test[['target', 'turbine']].copy()
for name, scores in test_scores.items(): predictions[name] = scores
predictions.to_parquet(OUTPUT / 'test_predictions.parquet')
print('Saved reports, figures and fitted models to', OUTPUT)


## What each model predicts over time
Run the cell below after evaluation. Each model gets a score plot with its decision threshold, correct warnings, false warnings and missed positive windows. The bottom panel shows the actual next-horizon target at each prediction time. Edit the turbine, split, start date and duration to explore without retraining.


In [ ]:
# Run after training and threshold evaluation. No models are retrained here.
# Edit these settings and rerun only this cell to inspect a different period.
PREDICTION_SPLIT = 'test'  # 'test' or 'validation'
PLOT_TURBINE = TURBINE_IDS[0]
PLOT_START = None  # None centres around the first positive; or '2024-04-01'.
PLOT_DURATION = pd.Timedelta(days=2)
PLOT_MODELS = None  # All five; or ['LightGBM', 'Random forest'].


def plot_prediction_timeline(split, turbine, window_start=None, duration=pd.Timedelta(days=2), names=None):
    import matplotlib.dates as mdates
    from matplotlib.lines import Line2D
    required = ('test_scores', 'validation_scores', 'thresholds', 'test', 'validation')
    if not all(key in globals() for key in required):
        raise RuntimeError('Run the data, training and threshold-evaluation cells first.')
    if split not in ('test', 'validation'):
        raise ValueError("Choose 'test' or 'validation'.")
    frame = test if split == 'test' else validation
    score_map = test_scores if split == 'test' else validation_scores
    timeline = frame[['target', 'turbine']].copy()
    for name, values in score_map.items():
        if len(values) != len(frame):
            raise ValueError('Scores and data differ. Rerun training/evaluation before plotting.')
        timeline[name] = np.asarray(values)
    timeline = timeline.loc[timeline.turbine.astype(str).eq(str(turbine))].sort_index()
    if timeline.empty:
        raise ValueError(f'No {split} predictions for turbine {turbine}.')
    if window_start is None:
        positive_times = timeline.index[timeline.target.eq(1)]
        centre = positive_times[0] if len(positive_times) else timeline.index[0]
        begin = max(timeline.index[0], centre - duration / 2)
    else:
        begin = pd.Timestamp(window_start)
    finish = begin + duration
    if pd.isna(begin) or pd.isna(finish) or finish <= begin:
        raise ValueError('Choose a valid date and positive duration.')
    selected = timeline.loc[(timeline.index >= begin) & (timeline.index < finish)]
    if selected.empty:
        raise ValueError(f'No eligible predictions here. Available: {timeline.index.min()} to {timeline.index.max()}')
    chosen = list(score_map) if names is None else list(names)
    if not chosen or any(name not in score_map for name in chosen):
        raise ValueError(f'Choose model names from {list(score_map)}')
    # Reindex only the display to avoid connecting lines over missing predictions.
    display_grid = pd.date_range(begin.floor('10min'), finish, freq=STEP, inclusive='left')
    shown = selected.reindex(display_grid)
    fig, axes = plt.subplots(len(chosen) + 1, 1, figsize=(16, 3 * len(chosen) + 2), sharex=True)
    comparison = selected[['target']].copy()
    for ax, name in zip(axes[:-1], chosen):
        values = selected[name]
        predicted = values.ge(thresholds[name])
        actual = selected.target.eq(1)
        ax.plot(shown.index, shown[name], color='tab:blue', linewidth=.8, label=f'Score: code(s) {ALARM_CODE}')
        ax.axhline(thresholds[name], color='purple', linestyle='--', label=f'Threshold {thresholds[name]:.4f}')
        for mask, colour, marker, label in [
            (predicted & actual, 'green', 'o', 'Correct warning (TP)'),
            (predicted & ~actual, 'darkorange', 'x', 'False warning (FP)'),
            (~predicted & actual, 'red', 'v', 'Missed alarm window (FN)')]:
            if mask.any():
                ax.scatter(selected.index[mask], values[mask], c=colour, marker=marker, s=35, label=label, zorder=4)
        ax.set(title=f'{name} | Code(s) {ALARM_CODE} | TP={int((predicted & actual).sum())}, FP={int((predicted & ~actual).sum())}, FN={int((~predicted & actual).sum())}',
               ylabel='Score (not calibrated)', ylim=(-.05, 1.05))
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
        ax.grid(alpha=.2)
        comparison[f'{name}: score'] = values
        comparison[f'{name}: predicted alarm'] = predicted.astype(int)
    axes[-1].step(shown.index, shown.target, where='post', color='black', linewidth=1)
    positive = selected.target.eq(1)
    axes[-1].scatter(selected.index[positive], selected.loc[positive, 'target'], color='red', s=25)
    axes[-1].set(yticks=[0, 1], ylim=(-.1, 1.2), ylabel='Actual target', xlabel='Prediction time t',
                 title=f'1 = alarm {ALARM_CODE} starts in [t, t + {HORIZON}); 0 = no new start')
    locator = mdates.AutoDateLocator()
    axes[-1].xaxis.set_major_locator(locator)
    axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    axes[-1].set_xlim(begin, finish)
    axes[-1].grid(alpha=.2)
    fig.suptitle(f'Turbine {turbine} | {split} | prior {LOOKBACK} -> next {HORIZON}\n' + alarm_plot_label(), fontsize=10, y=1.01)
    fig.tight_layout()
    plt.show()
    print('Every point is a prediction time, not an exact alarm-start timestamp.')
    print('A score at or above the dashed threshold predicts a new alarm within the horizon.')
    print('Gaps mean no eligible prediction. Class-weighted/random scores are not calibrated probabilities.')
    print('Detailed scores and 0/1 predictions (first 30 rows):')
    display(comparison.head(30))
    return comparison

prediction_window = plot_prediction_timeline(
    PREDICTION_SPLIT, PLOT_TURBINE, PLOT_START, PLOT_DURATION, PLOT_MODELS)


## Full training, validation and test prediction history
One panel per model shows scores across all three periods, with split shading, dashed boundaries and the decision threshold. Training scores are in-sample and must not be interpreted as held-out performance. Select HISTORY_TURBINE to inspect any turbine included in TURBINE_IDS; plotting a turbine does not train it. This cell only scores fitted models.


In [ ]:
# Full-history scores: run after training/evaluation. This does not retrain models.
HISTORY_TURBINE = TURBINE_IDS[0]  # Choose one of the turbines actually used for training.
HISTORY_MODELS = None  # All five; or ['LightGBM', 'Random forest'].


def plot_full_prediction_history(turbine, names=None):
    import matplotlib.dates as mdates
    needed = ('train', 'validation', 'test', 'models', 'thresholds',
              'validation_scores', 'test_scores', 'random_preprocessor', 'random_weights')
    if not all(key in globals() for key in needed):
        raise RuntimeError('Run the data, training and threshold-evaluation cells first.')
    chosen = list(test_scores) if names is None else list(names)
    if not chosen or any(name not in test_scores for name in chosen):
        raise ValueError(f'Choose model names from {list(test_scores)}')
    pieces = []
    for split_name, frame, stored in [('Training', train, None),
                                       ('Validation', validation, validation_scores),
                                       ('Test', test, test_scores)]:
        mask = frame.turbine.astype(str).eq(str(turbine)).to_numpy()
        part = frame.loc[mask, ['target', 'turbine']].copy()
        if part.empty:
            continue
        part['Split'] = split_name
        if stored is None:
            # Predict on training rows with already-fitted estimators, in batches.
            # These are IN-SAMPLE scores, not evidence of generalisation.
            x = frame.loc[mask, feature_columns]
            for name in chosen:
                batches = []
                for offset in range(0, len(x), 25000):
                    batch = x.iloc[offset:offset + 25000]
                    if name == 'Random-weight score':
                        z = np.clip(random_preprocessor.transform(batch) @ random_weights, -30, 30)
                        batches.append(1 / (1 + np.exp(-z)))
                    else:
                        batches.append(models[name].predict_proba(batch)[:, 1])
                part[name] = np.concatenate(batches)
        else:
            for name in chosen:
                if len(stored[name]) != len(frame):
                    raise ValueError('Stored scores do not match the split. Rerun training/evaluation.')
                part[name] = np.asarray(stored[name])[mask]
        pieces.append(part)
    if not pieces:
        raise ValueError(f'No predictions for turbine {turbine}; trained turbines: {TURBINE_IDS}')
    history = pd.concat(pieces).sort_index()
    grid = pd.date_range(history.index.min(), history.index.max(), freq=STEP)
    # Missing/purged periods remain NaN, so lines do not bridge data gaps.
    shown = history.reindex(grid)
    colors = {'Training': 'tab:blue', 'Validation': 'tab:orange', 'Test': 'tab:green'}
    first, last = grid[0], grid[-1] + STEP
    spans = [('Training', first, TRAIN_END), ('Validation', TRAIN_END, VALIDATION_END),
             ('Test', VALIDATION_END, last)]
    fig, axes = plt.subplots(len(chosen) + 1, 1,
                             figsize=(17, 2.8 * len(chosen) + 2), sharex=True)
    for ax, name in zip(axes[:-1], chosen):
        for split_name, _, _ in spans:
            values = shown[name].where(shown.Split.eq(split_name))
            ax.plot(grid, values, color=colors[split_name], linewidth=.5,
                    label=split_name + (' (in-sample)' if split_name == 'Training' else ''))
        ax.axhline(thresholds[name], color='purple', linestyle=':', linewidth=1,
                   label=f'Validation-selected threshold: {thresholds[name]:.4f}')
        ax.set(title=f'{name} | Code(s) {ALARM_CODE}', ylabel='Score', ylim=(-.05, 1.05))
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
    axes[-1].step(grid, shown.target, where='post', color='black', linewidth=.6)
    axes[-1].set(title=f'Actual outcome: alarm {ALARM_CODE} starts in the next {HORIZON}',
                 ylabel='Target', yticks=[0, 1], ylim=(-.1, 1.2), xlabel='Prediction time t')
    for ax in axes:
        for split_name, left, right in spans:
            left, right = max(first, left), min(last, right)
            if right > left:
                ax.axvspan(left, right, color=colors[split_name], alpha=.07, linewidth=0)
        for boundary in [TRAIN_END, VALIDATION_END]:
            if first <= boundary <= last:
                ax.axvline(boundary, color='black', linestyle='--', linewidth=1)
        ax.grid(alpha=.2)
        ax.set_xlim(first, last)
    for boundary, label in [(TRAIN_END, 'Train ends / validation starts'),
                             (VALIDATION_END, 'Validation ends / test starts')]:
        if first <= boundary <= last:
            axes[0].annotate(label + f' ({boundary:%Y-%m-%d})',
                             xy=(boundary, 1), xycoords=('data', 'axes fraction'),
                             xytext=(4, 6), textcoords='offset points', fontsize=8)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    fig.suptitle(f'Turbine {turbine} | full history | previous {LOOKBACK} -> next {HORIZON}\n' + alarm_plot_label(), fontsize=10, y=1.02)
    fig.tight_layout()
    plt.show()
    print('Training scores are in-sample; validation chose thresholds; test measures later-data performance.')
    print('Shading marks split date ranges. Gaps mean no eligible predictions (including missing years and purged windows).')
    print('Scores are not calibrated probabilities. Each point still predicts only the configured horizon.')
    display(history.groupby('Split', sort=False).agg(
        Predictions=('target', 'size'), Actual_positive_windows=('target', 'sum')))
    return history

full_history_predictions = plot_full_prediction_history(HISTORY_TURBINE, HISTORY_MODELS)
